# 现代 LLM Serving System：从 Continuous Batching 到 PD 分离

> 前四章一直站在“单请求”视角：
>
> - 20：Logits 怎么变成 Token？
> - 21：一次 Token 为什么慢？
> - 22：怎么减少权重 / Activation / KV 的位宽？
> - 23：怎么让一次 Target Forward 确认多个 Token？
>
> 现在切换到生产环境。
>
> 假设一张 GPU 同时来了 100 个请求：
>
> ```text
> A: prompt=20,    output=50
> B: prompt=20000, output=100
> C: prompt=800,   output=2000
> D: prompt=4000,  output=20
> ...
> ```
>
> 单个请求再快，也不能回答这些问题：
>
> - 谁先算？
> - 谁和谁放在同一个 Batch？
> - 一个请求结束后，空出来的位置能不能马上给新请求？
> - 不同长度的 KV Cache 放在哪里？
> - 很多请求共享相同 System Prompt，能不能复用 Prefill？
> - 一个超长 Prompt 会不会卡住一堆正在 Decode 的请求？
> - Prefill 和 Decode 的硬件瓶颈不同，为什么一定要放在同一台 GPU？
>
> 本章就是一张 **现代 LLM Serving 黑话地图**。
>
> 目标不是背名词，而是做到：看到 `Continuous Batching / PagedAttention / Prefix Caching / RadixAttention / Chunked Prefill / PD Disaggregation / FlashInfer / CUDA Graph / TP / PP / DP / EP / CP` 时，知道它到底在解决哪一个资源问题。

先从最容易感受到的服务指标开始。

## 1. Throughput、TTFT、TPOT、ITL：先说清楚“快”是什么意思

线上 Serving 常见四类指标：

| 指标 | 含义 | 用户/系统关心什么 |
|---|---|---|
| Throughput | 单位时间总 Token 数 / 请求数 | GPU 总产能 |
| TTFT | 请求到首 Token | 用户“多久开始看到回复” |
| TPOT / ITL | 后续 Token 间隔 | 用户“回复刷得快不快” |
| Concurrency | 同时活跃请求数 | 服务容量 |

一个系统可能：

```text
吞吐很高
但单个用户 TPOT 很差
```

也可能：

```text
TTFT 很低
但高并发时吞吐不够
```

所以读厂商报告时，第一件事不是看“2.3x”，而是看它优化的究竟是哪一个指标。

In [ ]:
requests = [
    {"arrival":0.0, "first":0.8, "finish":2.0, "tokens":25},
    {"arrival":0.1, "first":0.5, "finish":1.5, "tokens":20},
    {"arrival":0.2, "first":1.0, "finish":2.4, "tokens":30},
]

ttfts = [r["first"] - r["arrival"] for r in requests]
total_tokens = sum(r["tokens"] for r in requests)
wall_time = max(r["finish"] for r in requests) - min(r["arrival"] for r in requests)

print("平均 TTFT:", sum(ttfts)/len(ttfts))
print("整体 throughput:", total_tokens/wall_time, "tokens/s")

这两个数描述的是完全不同的东西。

后面每讲一个系统技术，都应该回到这里问：

> 它主要改善 TTFT、TPOT、Throughput、显存容量，还是 Tail Latency？

这比背“某技术能加速”更重要。

## 2. Static Batching：为什么短请求要陪长请求一起等？

先看最朴素的 Batch。

GPU 一次最多放 4 个请求：

```text
Batch 1: [5, 200, 8, 150]
Batch 2: [6, 180, 10, 120]
```

这里数字表示每个请求还要生成多少 Decode Token。

如果是 Static Batching：

```text
4 个请求一起进入
→ 必须等这一批都结束
→ 才能换下一批
```

第一批里 5-token 请求早就结束了，但它占的 slot 不能给后面的请求。

这就是资源空洞。

In [ ]:
requests = [5, 200, 8, 150, 6, 180, 10, 120]
batch_size = 4

def static_batch_steps(requests, batch_size):
    total = 0
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        total += max(batch)
    return total

print("Static Batching 总 iteration:", static_batch_steps(requests, batch_size))

这里不是说真实 GPU 时间就等于 iteration 数。

这个 toy model 只想暴露一个问题：

> **请求完成后，为什么不能立即退出；新请求为什么不能立即补进来？**

解决它的就是 Continuous Batching。

## 3. Continuous Batching：Batch 不再是一批“固定成员”

Continuous Batching 也叫：

- In-flight Batching
- Iteration-level Scheduling

核心不是“Batch Size 变大”，而是：

> **每个 Decode Iteration 都允许请求进入和退出。**

```text
slot 1: A 剩 5 token
slot 2: B 剩 200
slot 3: C 剩 8
slot 4: D 剩 150

A 完成
↓
下一个 iteration 立刻把 E 塞进 slot 1
```

我们写一个带有限 capacity 的模拟器。

In [ ]:
from collections import deque

def continuous_batch_steps(requests, capacity=4):
    pending = deque(requests)
    active = []
    steps = 0

    while pending or active:
        while pending and len(active) < capacity:
            active.append(pending.popleft())

        # 当前 iteration，每个 active request decode 1 token
        active = [x - 1 for x in active]
        active = [x for x in active if x > 0]
        steps += 1

    return steps

print("Static:", static_batch_steps(requests, 4))
print("Continuous:", continuous_batch_steps(requests, 4))

这才是 Continuous Batching 的核心：

```text
Static:
成员固定到整批结束

Continuous:
每个 iteration 重组 active batch
```

以后在 vLLM / SGLang JD 里看到：

```text
scheduler
iteration-level scheduling
max-num-seqs
continuous batching
```

它们都在这一层。

但请求可以灵活进出了，下一问题来了：

> **每个请求的 KV Cache 长度都不一样，显存怎么放？**

## 4. KV Cache 的“外部碎片”：为什么连续显存块不好管理？

假设要提前为每个请求预留最大长度：

```text
Request A 实际 100 tokens，但预留 4096
Request B 实际 3500 tokens，预留 4096
Request C 实际 50 tokens，但预留 4096
```

会浪费大量空间。

如果每个请求都申请一整块连续 KV Cache，又会遇到：

- 长度不断增长
- 请求随时进入 / 离开
- 不同请求长度差异巨大
- 连续大块很难找到

这和操作系统管理内存很像。

于是 vLLM 的经典设计 **PagedAttention** 出现了。

## 5. PagedAttention：KV Cache 也做“分页”

PagedAttention 的直觉：

```text
不要给一个 Request 预留一整块连续大显存
↓
把 KV Cache 切成固定大小 Block / Page
↓
逻辑上连续
物理上可以分散
```

类似虚拟内存：

```text
Request A logical blocks: 0 1 2 3
physical GPU blocks:     8 2 19 5
```

系统维护一张 Block Table，把逻辑 Token 位置映射到物理 KV Block。

因此：

- 请求增长时，只再申请新 Block。
- 请求结束时，Block 立即归还 Pool。
- 不要求整段 KV 物理连续。
- 更容易提高显存利用率。

In [ ]:
import math

def block_allocation(seq_len, block_size):
    blocks = math.ceil(seq_len / block_size)
    allocated_tokens = blocks * block_size
    waste = allocated_tokens - seq_len
    return blocks, waste

for seq in [17, 63, 129, 1025]:
    blocks, waste = block_allocation(seq, block_size=16)
    print(f"seq={seq:4d}: blocks={blocks:3d}, 最后一页浪费={waste:2d} token slots")

PagedAttention 并没有让 KV Cache “消失”。

它解决的是：

> **怎么把不断增长、长度不同的 KV Cache 更灵活地分配。**

所以看到：

```text
PagedAttention
KV block manager
block table
page size
```

先想到 **KV Memory Management**，不要先想到 Attention 数学公式。

## 6. Prefix Caching：相同前缀为什么还要重复 Prefill？

很多线上请求共享前缀：

```text
System Prompt
公司知识库说明
工具 Schema
Few-shot 示例
...
```

如果 100 个请求前 4000 Token 完全一样，每次都重新 Prefill 一遍很浪费。

**Prefix Caching** 的思路：

```text
已经算过的 Prefix KV
↓
缓存起来
↓
下一个拥有相同 Prefix 的请求直接复用
```

它主要减少重复 Prefill 工作，因此常能改善 TTFT 和 Prefill 吞吐。

In [ ]:
shared_prefix = 4000
requests = 100

without_cache = shared_prefix * requests
with_cache = shared_prefix  # 简化：第一次算一次

print("无 Prefix Cache，重复处理 token:", without_cache)
print("理想 Prefix Cache，只处理一次:", with_cache)
print("理论重复工作减少倍数:", without_cache / with_cache)

真实系统当然还要处理：

- Prefix 是否完全一致
- Block 边界
- Cache eviction
- Cache hit rate
- 多租户安全
- Hash / tree indexing

SGLang 的 **RadixAttention** 就是这一类问题里非常典型的名字。

## 7. RadixAttention：为什么 Prefix Cache 还需要一棵树？

如果请求之间不是“全前缀一致”，而是分叉：

```text
System Prompt
├── 用户 A 的对话历史
│   ├── A1
│   └── A2
└── 用户 B 的对话历史
    ├── B1
    └── B2
```

我们希望不同请求能共享任意长度的公共 Prefix。

Radix Tree / Prefix Tree 很适合表达：

```text
共同前缀
→ 分叉
→ 再共享
```

SGLang 用 **RadixAttention** 把 Prefix KV Cache 和 Radix Tree 管理结合起来。

看到这个词时，最重要的 mental model 是：

> **它是 KV Prefix Reuse + Cache Management，不是新的 Attention 公式。**

## 8. 一个超长 Prefill 为什么会卡住 Decode？

假设 GPU 正在服务很多 Decode 请求，每个请求都希望每 40 ms 出一个 Token。

突然来了一个 100k Prompt。

如果这个 Prefill 一口气占住 GPU 很久：

```text
长 Prefill
████████████████████████████████
Decode 请求：等、等、等……
```

TPOT / ITL 会突然恶化。

这类问题叫 **Head-of-Line Blocking / Prefill Interference**。

一个思路是：

> 不要一次吞完整个长 Prompt，把 Prefill 切成小块。

## 9. Chunked Prefill：把长 Prompt 切成块和 Decode 交错

Chunked Prefill：

```text
100k Prompt
↓
chunk 1
chunk 2
chunk 3
...
```

Scheduler 可以把：

```text
一部分 Prefill Token
+
一批 Decode Token
```

放进同一个调度周期。

目标是：

- 避免超长 Prefill 独占 GPU
- 控制 TTFT / TPOT trade-off
- 更灵活地填满 token budget

所以看到：

```text
chunked prefill
max-num-batched-tokens
token budget
prefill chunk
```

应该想到 **Scheduler 在平衡 Prefill 和 Decode**。

In [ ]:
prompt_len = 100_000
for chunk in [512, 2048, 8192]:
    chunks = math.ceil(prompt_len / chunk)
    print(f"chunk={chunk:5d} -> 100k prompt 被切成 {chunks} 个 prefill chunks")

Chunk 越小不一定越好：

```text
chunk 小
→ 更容易插入 Decode
→ 但调度 / launch overhead 更多

chunk 大
→ Prefill 更连续
→ 但可能拖慢 Decode
```

所以系统调优里经常没有“一条参数适合所有 workload”。

## 10. PD 分离：既然 Prefill 和 Decode 不一样，为什么还用同一批 GPU？

到这里最重要的“厂商报告黑话”出现了：

**Prefill-Decode Disaggregation / PD Disaggregation / PD Separation**

前面已经知道：

```text
Prefill
- 一次处理很多 Prompt Token
- 更偏 compute-intensive
- 关心 TTFT

Decode
- 每步一个 Token
- 更偏 memory-bandwidth / latency-sensitive
- 关心 TPOT / ITL
```

既然两阶段资源特征不同，一个自然想法就是：

```text
Prefill Workers
      ↓ 生成 KV
KV Transfer
      ↓
Decode Workers
```

这就是 PD 分离的基本拓扑。

PD 分离解决的不是一个单点 kernel 问题，而是**资源池和调度问题**。

它允许：

- Prefill Pool 单独扩容。
- Decode Pool 单独扩容。
- 针对 TTFT / TPOT 分别做 SLO。
- 不同阶段使用不同 GPU 配置。
- 避免长 Prefill 直接干扰 Decode。

但它引入新成本：

```text
KV Cache 必须从 Prefill Worker 传给 Decode Worker
```

所以会继续看到：

- KV Transfer
- KV Connector
- RDMA
- NCCL
- NIXL
- disaggregated serving
- remote KV cache

这些都不是凭空冒出来的，它们是 **PD 分离后的数据移动问题**。

## 11. FlashAttention 和 FlashInfer：它们在 Stack 的哪一层？

前面讲的很多技术都属于 Scheduler / Memory Manager。

但底层 Kernel 也很重要。

### FlashAttention

核心关键词：

```text
IO-aware
tiling
减少 HBM 读写
避免显式 materialize 大 Attention Matrix
```

长 Prompt Prefill 时尤其重要。

### FlashInfer

更偏 LLM Serving 场景的高性能 Kernel / Primitive，包括 Attention、Sampling 等。

所以可以把它们放在：

```text
Serving Engine
├── Scheduler
├── KV Cache Manager
└── Kernel Backend
    ├── FlashAttention
    ├── FlashInfer
    └── vendor kernels
```

看到招聘 JD 写：

```text
CUDA / Triton / FlashAttention / FlashInfer
```

通常已经从“系统调度”下钻到“Kernel 优化”这一层。

## 12. CUDA Graph：为什么 Kernel Launch 本身也会成为开销？

Decode 每步计算规模比较小，而且要重复很多次。

如果每一步都让 CPU：

```text
准备 kernel
launch kernel
同步
再 launch
```

Launch overhead 可能变得明显。

**CUDA Graph** 可以把一段 GPU 工作捕获下来，后续重复 replay，减少 CPU launch overhead。

它特别适合：

```text
形状相对稳定
重复执行
```

的 Decode 路径。

所以：

> CUDA Graph 解决的不是 FLOPs，也不是 KV Cache，而是 **GPU launch / orchestration overhead**。

## 13. TP / PP / DP / EP / CP：多 GPU 以后，到底在“并行”什么？

模型单卡放不下，或者单卡吞吐不够，就进入 Distributed Inference。

这些缩写非常常见。

### TP — Tensor Parallel

把一个大矩阵切到多张 GPU。

```text
同一层
→ 多 GPU 一起算
```

优点：能拆大模型。  
代价：每层频繁通信。

### PP — Pipeline Parallel

把不同层放到不同 GPU。

```text
GPU 0: layer 0~15
GPU 1: layer 16~31
```

### DP — Data Parallel / Replica

多份模型副本，各服务不同请求。

```text
Replica 0 ← requests A/B
Replica 1 ← requests C/D
```

### EP — Expert Parallel

MoE 模型把不同 Expert 分到不同 GPU。

### CP — Context Parallel

针对超长 Context，把序列 / Attention 相关工作分到多 GPU。

以后看到：

```text
TP=8, PP=2, EP=8
```

先问：

> **它把模型的哪一个维度切开了？**

## 14. 把厂商“吹牛名词”放回一张图

```text
Incoming Requests
       ↓
Admission / Queue
       ↓
Scheduler
├─ Continuous Batching
├─ Chunked Prefill
└─ Prefill / Decode policy
       ↓
KV Cache Manager
├─ PagedAttention / Blocks
├─ Prefix Caching
└─ RadixAttention
       ↓
Model Execution
├─ Quantization
├─ Speculative Decoding
├─ CUDA Graph
└─ FlashAttention / FlashInfer
       ↓
Distributed
├─ TP / PP / DP
├─ EP / CP
└─ PD Disaggregation + KV Transfer
       ↓
Tokens / Streaming API
```

以后不管厂商给一个多新的名字，都先问：

1. 它在这张图哪一层？
2. 它减少计算、减少内存、减少通信，还是改善调度？
3. 它改善 TTFT、TPOT、Throughput 还是 capacity？
4. 它增加了什么新代价？

## 15. 一个报告怎么读：不要只看 “x2.1 throughput”

假设报告写：

> “Our new serving stack achieves 2.1x throughput.”

应该继续追问：

```text
模型？
GPU？
精度？
Prompt length distribution？
Output length？
Concurrency？
TTFT SLO？
TPOT SLO？
P50 / P95 / P99？
是否使用 Prefix Cache？
是否用 Speculative Decoding？
是否允许 TTFT 变差换 Throughput？
```

Serving Benchmark 的数字高度依赖 workload。

真正的能力不是背某个“世界纪录”，而是能判断比较是否公平。

## 小结

把这一章按问题记，而不是按名字记：

- **请求长短不一，Batch slot 浪费** → Continuous Batching
- **KV 长度不同、连续显存难分配** → PagedAttention
- **大量请求共享 Prefix** → Prefix Caching / RadixAttention
- **超长 Prompt 干扰 Decode** → Chunked Prefill
- **Prefill / Decode 硬件特征不同** → PD 分离
- **PD 分离后 KV 要跨 Worker** → KV Transfer / RDMA / Connector
- **Attention IO 太大** → FlashAttention / FlashInfer
- **Decode launch overhead** → CUDA Graph
- **单卡放不下 / 算不过来** → TP / PP / DP / EP / CP

下一章我们会问：

> 系统变快以后，模型质量有没有掉？不同方案怎么公平比较？

## 作业

1. 写一个 `capacity=8` 的 Continuous Batching 模拟器，并记录每个请求完成时间。
2. 用 block_size=16 计算 seq_len `[17,33,100,1025]` 的 page 内部浪费。
3. 解释 Prefix Caching 和 RadixAttention 的关系。
4. 用一句话解释 Chunked Prefill 为什么可能改善 TPOT。
5. 画出 PD 分离的最小拓扑，并标出 KV Transfer。
6. 分别解释 TP、PP、DP、EP 在“切什么”。
7. 找一篇任意推理厂商报告，把里面的 5 个名词放回本章 Serving Stack。